# **Audio and Sample Rate Checks with Librosa**

In [1]:
# Loading audio & checking sample rate

import os
import pathlib
import librosa
import numpy as np

print(pathlib.Path.cwd())
print("Imports successful!")

c:\Users\winni\miniconda3\envs\DS_class\Lib\site-packages\librosa\util\files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


c:\Users\winni\music-genre-class\music-genre-classification\Data_Music
Imports successful!


In [2]:
# Testing file path
test_path = "genres_original/blues/blues.00000.wav"
print(os.path.isfile(test_path)) # Must print true

True


In [3]:
# Passing target SR (sample rate) directly to have Librosa resample on load

TARGET_SR = 22050  # GTZAN standard

def load_audio(file_path, target_sr=TARGET_SR):
    """Load a wav file; returns (signal, sample_rate) or (None, None) on failure."""
    try:
        y, sr = librosa.load(file_path, sr=target_sr, mono=True)
        return y, sr
    except Exception as e:
        print(f"FAILED to load {file_path}: {e}")
        return None, None

y, sr = load_audio("genres_original/blues/blues.00000.wav")

if y is None:
    print("Audio failed to load — check the file path and sndfile installation.")
else:
    print(f"Signal shape: {y.shape}, Sample rate: {sr}")
    # Signal shape: (661500,), Sample rate: 22050

Signal shape: (661794,), Sample rate: 22050


#### **Extracting MFCCs**

In [4]:
# Extracting MFCCs (Mel-Frequency Cepstral Coefficients) as features for genre classification

N_MFCC = 40  # 40 coefficients is a strong default for genre classification

def extract_mfcc(y, sr, n_mfcc=N_MFCC):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    # mfcc shape: (n_mfcc, time_frames) → e.g. (40, 1292) for a 30s clip at sr=22050
    return mfcc

mfcc = extract_mfcc(y, sr)
print(f"MFCC shape: {mfcc.shape}")

MFCC shape: (40, 1293)


#### **Extracting Mel Spectrograms**

In [5]:
# Extracting Mel spectrogram as an alternative feature representation

def extract_mel_spectrogram(y, sr, n_mels=128, n_fft=2048, hop_length=512):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)  # convert to dB scale
    return mel_db

mel = extract_mel_spectrogram(y, sr)
print(f"Mel spectrogram shape: {mel.shape}")

Mel spectrogram shape: (128, 1293)


#### **MFCC vs Mel Spectrogram**

MFCCs are compact (40xT) and encode timbral texture, which is needed for genre tasks. Mel Spectrograms (128xT) preserve more spectral detail and work well with CNN architectures. Team will review and determine most viable option for the model.